# Import Required Libraries

In [7]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import joblib


# Load the Dataset

In [8]:
df=pd.read_csv("/content/updated_telco_churn_with_id.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1.0,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34.0,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2.0,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45.0,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2.0,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [9]:
df.isnull().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,364
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7395 entries, 0 to 7394
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7395 non-null   object 
 1   gender            7395 non-null   object 
 2   SeniorCitizen     7395 non-null   int64  
 3   Partner           7395 non-null   object 
 4   Dependents        7395 non-null   object 
 5   tenure            7031 non-null   float64
 6   PhoneService      7395 non-null   object 
 7   MultipleLines     7395 non-null   object 
 8   InternetService   7395 non-null   object 
 9   OnlineSecurity    7395 non-null   object 
 10  OnlineBackup      7395 non-null   object 
 11  DeviceProtection  7395 non-null   object 
 12  TechSupport       7395 non-null   object 
 13  StreamingTV       7395 non-null   object 
 14  StreamingMovies   7395 non-null   object 
 15  Contract          7395 non-null   object 
 16  PaperlessBilling  7395 non-null   object 


# Remove the unncessary columns

In [11]:
cols_to_remove = [
    "customerID",
    "Partner",
    "Dependents",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "StreamingTV",
    "StreamingMovies",
    "PaperlessBilling"
]
df = df.drop(columns=[col for col in cols_to_remove if col in df.columns])

In [12]:
df.duplicated().sum()

np.int64(394)

In [13]:
df.drop_duplicates(inplace=True)

In [15]:
df['Churn'].value_counts()

,count
Churn,
No,5154
Yes,1847


# Separate Features and Target

In [20]:
X=df.drop('Churn',axis=1)
y=df['Churn']

# Identify Numerical and Categorical Columns

In [21]:
num_cols = X.select_dtypes(include=['int64', 'float64']).columns

In [22]:
cat_cols = X.select_dtypes(include=['object']).columns

In [23]:
print(num_cols)
print(cat_cols)

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges'], dtype='object')
Index(['gender', 'PhoneService', 'InternetService', 'TechSupport', 'Contract',
       'PaymentMethod', 'TotalCharges'],
      dtype='object')


# Split the Dataset

In [24]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=42,stratify=y)

In [26]:
X_test

,gender,SeniorCitizen,tenure,PhoneService,InternetService,TechSupport,Contract,PaymentMethod,MonthlyCharges,TotalCharges
2694,Male,0,8.0,Yes,No,No internet service,Month-to-month,Credit card (automatic),19.65,169.75
8,Female,0,28.0,Yes,Fiber optic,Yes,Month-to-month,Electronic check,104.80,3046.05
1354,Female,0,18.0,Yes,Fiber optic,Yes,Month-to-month,Electronic check,88.35,1639.3
6851,Male,0,38.0,Yes,Fiber optic,Yes,Two year,Bank transfer (automatic),94.90,3616.25
3314,Female,0,37.0,Yes,Fiber optic,No,One year,Electronic check,80.05,3019.1
...,...,...,...,...,...,...,...,...,...,...
2142,Female,0,21.0,Yes,DSL,No,One year,Mailed check,64.85,1336.8
3006,Male,1,14.0,Yes,Fiber optic,No,Month-to-month,Electronic check,74.95,1036.75
3685,Male,0,50.0,No,DSL,Yes,One year,Bank transfer (automatic),44.45,2188.45
1449,Female,1,6.0,Yes,No,No internet service,Month-to-month,Bank transfer (automatic),19.50,106.8


# Create Numerical Preprocessing Pipeline

In [27]:
numerical_pipeline=Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='mean')),
    ('scaler',StandardScaler())
])


# Create Categorical Preprocessing Pipeline

In [28]:
categorical_pipeline=Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoder',OneHotEncoder(handle_unknown='ignore'))
])

# Combine Preprocessing using ColumnTransformer

In [29]:
preprocessing=ColumnTransformer(transformers=[
    ('num',numerical_pipeline,num_cols),
    ('cat',categorical_pipeline,cat_cols)
])


# Create the Final Reusable ML Pipeline

In [30]:
model_pipeline=Pipeline(steps=[
    ('preprocesser',preprocessing),
    ('classifier',LogisticRegression(max_iter=1000))
])

# Train the Pipeline

In [31]:
model_pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocesser',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['SeniorCitizen', 'tenure', 'MonthlyCharges'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['gender', 'PhoneService', 'InternetService', 'TechSupport', 'Contract',
       'PaymentMethod', 'TotalCharges'],
      dtype='object'))])),
                ('classifier', LogisticRegression(max_iter=1000))])

# Make Predictions

In [32]:
y_pred=model_pipeline.predict(X_test)

In [37]:
y_pred

array(['No', 'No', 'Yes', ..., 'No', 'No', 'No'], dtype=object)

# Evaluate the Model

In [33]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("Classification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8098229583095374
Confusion Matrix:
[[1175  114]
 [ 219  243]]
Classification Report:
              precision    recall  f1-score   support

          No       0.84      0.91      0.88      1289
         Yes       0.68      0.53      0.59       462

    accuracy                           0.81      1751
   macro avg       0.76      0.72      0.73      1751
weighted avg       0.80      0.81      0.80      1751



# Save the Reusable Pipeline

In [34]:
joblib.dump(model_pipeline, "customer_churn_pipeline.pkl")

['customer_churn_pipeline.pkl']

# Load and Reuse the Pipeline

In [35]:
loaded_pipeline = joblib.load("customer_churn_pipeline.pkl")

# Predict for New Customer Data

In [44]:
new_customer=pd.DataFrame({
    "gender":['Male'],
    "SeniorCitizen":[0],
    "tenure":[50],
    "PhoneService":['Yes'],
    "InternetService":['Fiber optic'],
    "TechSupport":['Yes'],
    "Contract":['Two year'],
    "PaymentMethod":['Bank transfer (automatic)	'],
    "MonthlyCharges":[100],
    "TotalCharges":[1200]
})
prediction=loaded_pipeline.predict(new_customer)
print("Prediction:",prediction)

Prediction: ['No']
